<a href="https://colab.research.google.com/github/AleR26/ColabFiles/blob/main/titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instalar PySpark en Google Colab
# PySpark no viene preinstalado en Google Colab.
# Se instala una única vez por sesión — si el runtime se reinicia,
# hay que volver a ejecutar esta celda.
# Código para Verificar si PySpark ya está instalado
try:
    import pyspark
    print("PySpark ya está instalado")
    print("Versión:", pyspark.__version__)

except ModuleNotFoundError:
    print("PySpark no está instalado. Instalando...")
    !pip install pyspark -q

    import pyspark
    print("Instalación completada")
    print("Versión:", pyspark.__version__)


PySpark ya está instalado
Versión: 4.0.2


In [2]:
from pyspark.sql import SparkSession

# ── Crear SparkSession con el patrón Builder ────────────────────────────────────
spark = (
    SparkSession.builder

    # Nombre visible en la Spark UI y en los logs del clúster
    .appName('Spark SQL: Dataframe')

    # local[*] = modo local usando TODOS los cores disponibles
    # En producción: 'yarn', 'k8s://...', 'spark://<host>:7077'
    .master('local[*]')


    # Número de particiones en operaciones de shuffle (default: 200)
    # 200 particiones para datasets pequeños es excesivo y lento
    # Regla general: 2-4 particiones por core en el clúster
    .config('spark.sql.shuffle.partitions', '8')

    # Adaptive Query Execution: Spark ajusta el plan de ejecución
    # en tiempo real según el tamaño real de los datos (Spark 3.0+)
    .config('spark.sql.adaptive.enabled', 'true')

    # Si ya existe una sesión activa, la reutiliza (no crea una nueva)
    .getOrCreate()
)

print(f'   SparkSession lista')
print(f'   Versión Spark    : {spark.version}')
print(f'   Nombre app       : {spark.sparkContext.appName}')
print(f'   Master           : {spark.sparkContext.master}')
print(f'   Cores disponibles: {spark.sparkContext.defaultParallelism}')

   SparkSession lista
   Versión Spark    : 4.0.2
   Nombre app       : Spark SQL: Dataframe
   Master           : local[*]
   Cores disponibles: 2


In [4]:
df = spark.read.csv("/content/drive/MyDrive/Titanic-Dataset (1).csv", header=True, inferSchema=True)

df.printSchema()

df.show()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|   

In [6]:
df.select("Name","Age","Sex").show()

+--------------------+----+------+
|                Name| Age|   Sex|
+--------------------+----+------+
|Braund, Mr. Owen ...|22.0|  male|
|Cumings, Mrs. Joh...|38.0|female|
|Heikkinen, Miss. ...|26.0|female|
|Futrelle, Mrs. Ja...|35.0|female|
|Allen, Mr. Willia...|35.0|  male|
|    Moran, Mr. James|NULL|  male|
|McCarthy, Mr. Tim...|54.0|  male|
|Palsson, Master. ...| 2.0|  male|
|Johnson, Mrs. Osc...|27.0|female|
|Nasser, Mrs. Nich...|14.0|female|
|Sandstrom, Miss. ...| 4.0|female|
|Bonnell, Miss. El...|58.0|female|
|Saundercock, Mr. ...|20.0|  male|
|Andersson, Mr. An...|39.0|  male|
|Vestrom, Miss. Hu...|14.0|female|
|Hewlett, Mrs. (Ma...|55.0|female|
|Rice, Master. Eugene| 2.0|  male|
|Williams, Mr. Cha...|NULL|  male|
|Vander Planke, Mr...|31.0|female|
|Masselmani, Mrs. ...|NULL|female|
+--------------------+----+------+
only showing top 20 rows


In [12]:
df.filter((df.Age > 18) & (df.Sex == "female")).show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----------+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|      Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----------+--------+
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|        C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925|       NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1|       C123|       S|
|          9|       1|     3|Johnson, Mrs. Osc...|female|27.0|    0|    2|          347742|11.1333|       NULL|       S|
|         12|       1|     1|Bonnell, Miss. El...|female|58.0|    0|    0|          113783|  26.55|       C103|       S|
|         16|       1|     2|Hew

In [14]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

In [19]:
df.select("Pclass").distinct().show()

+------+
|Pclass|
+------+
|     3|
|     1|
|     2|
+------+



In [22]:
total_p = df.select("Pclass").count()
total_s = df.select("Sex").count()
print("Total de pasajeros contados con Pclass: ", total_p)
print("Total de pasajeros con columna Sex: ", total_s)

Total de pasajeros contados con Pclass:  891
Total de pasajeros con columna Sex:  891
